# ĐỒ ÁN BIG DATA - NHÓM 11
## Đề tài: Phân tích hành vi mua sắm & Gợi ý sản phẩm mua kèm (Market Basket Analysis)
- **GVHD:** TS. Ngô Dương Hà
- **Sinh viên thực hiện:** Trần Thị Kim Ngân (2001230554), Lê Tuấn Hội (2001230278)
- **Hệ thống:** MongoDB NoSQL + Apache Spark FP-Growth trên Google Colab

### Bước 1: Cài đặt Java, MongoDB Server và PySpark trên Google Colab
*Lưu ý: Colab chạy Ubuntu 22.04 nên cần cài đặt gói MongoDB 7.0 chính thức và khởi động dạng daemon với `--fork`.*

In [ ]:
# 1. Cài đặt Java JDK và các thư viện hệ thống cần thiết
!apt-get update -qq > /dev/null
!apt-get install -y default-jdk-headless libcurl4 openssl -qq > /dev/null

# 2. Cài đặt MongoDB 7.0 chính thức cho Ubuntu 22.04 trên Colab
!curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor --yes
!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-7.0.list
!apt-get update -qq > /dev/null
!apt-get install -y mongodb-org -qq > /dev/null

# 3. Tạo thư mục dữ liệu và khởi chạy daemon mongod
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db --bind_ip 127.0.0.1

# 4. Cài đặt các thư viện Python: PySpark, PyMongo, Pandas
!pip install pyspark pymongo pandas -q

# 5. Kiểm tra kết nối MongoDB ngay lập tức để đảm bảo sẵn sàng
import time
from pymongo import MongoClient
time.sleep(2)
try:
    client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=3000)
    client.server_info()
    print("\n[+] MongoDB Server đã khởi động và kết nối thành công 100% trên cổng 27017!")
    print("[+] Môi trường Java và PySpark đã sẵn sàng!")
except Exception as e:
    print("\n[-] Không thể kết nối MongoDB:", e)

### Bước 2: Kết nối Google Drive & Xác thực đường dẫn bộ dữ liệu
*(Dữ liệu nằm tại thư mục `My Drive > InstacartMarketBasketAnalysis`)*

In [ ]:
import os
import glob
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Đường dẫn tới thư mục dữ liệu trên Google Drive
DATA_DIR = '/content/drive/MyDrive/InstacartMarketBasketAnalysis'

# Tự động phát hiện nếu tên thư mục có sai khác nhỏ
if not os.path.exists(DATA_DIR):
    matches = glob.glob('/content/drive/MyDrive/*Instacart*')
    if matches:
        DATA_DIR = matches[0]

print(f"Đang sử dụng thư mục dữ liệu: {DATA_DIR}")
print("\nDanh sách các file trong thư mục:")
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.csv'):
        fsize = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024 * 1024)
        print(f"  - {f:<28} ({fsize:.2f} MB)")

### Bước 3: Nạp dữ liệu vào MongoDB theo chuẩn Document Model

In [ ]:
import time
import pandas as pd
from pymongo import MongoClient

# Kết nối tới MongoDB cục bộ trên Colab
client = MongoClient("mongodb://localhost:27017/")
db = client["instacart_bigdata"]

# -----------------------------------------------------
# 3.1. Nạp danh mục Products
# -----------------------------------------------------
print("--- [1/2] Nạp Collection 'products' ---")
df_products = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
df_aisles = pd.read_csv(os.path.join(DATA_DIR, "aisles.csv"))
df_departments = pd.read_csv(os.path.join(DATA_DIR, "departments.csv"))

df_merged = df_products.merge(df_aisles, on="aisle_id", how="left")
df_merged = df_merged.merge(df_departments, on="department_id", how="left")

col_products = db["products"]
col_products.drop()

prod_records = []
for _, row in df_merged.iterrows():
    prod_records.append({
        "_id": int(row["product_id"]),
        "product_id": int(row["product_id"]),
        "product_name": str(row["product_name"]),
        "aisle": str(row["aisle"]),
        "department": str(row["department"])
    })

col_products.insert_many(prod_records)
col_products.create_index("product_name")
print(f"[+] Đã nạp thành công {col_products.count_documents({}):,} sản phẩm vào MongoDB.")

# -----------------------------------------------------
# 3.2. Nạp giỏ hàng Orders (Gom nhóm thành Document)
# -----------------------------------------------------
print("\n--- [2/2] Nạp Collection 'orders' (Transactions) ---")
prod_dict = dict(zip(df_products["product_id"], df_products["product_name"]))

df_order_prod = pd.read_csv(os.path.join(DATA_DIR, "order_products__train.csv"))
df_orders = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))

# Lọc các order tồn tại trong tập train
valid_order_ids = set(df_order_prod["order_id"].unique())
df_orders = df_orders[df_orders["order_id"].isin(valid_order_ids)]

orders_meta = {}
for _, row in df_orders.iterrows():
    orders_meta[int(row["order_id"])] = {
        "user_id": int(row["user_id"]),
        "order_dow": int(row["order_dow"]),
        "order_hour_of_day": int(row["order_hour_of_day"])
    }

# Gom sản phẩm theo từng giỏ hàng
baskets = {}
for _, row in df_order_prod.iterrows():
    oid = int(row["order_id"])
    pname = prod_dict.get(int(row["product_id"]), f"Product_{row['product_id']}")
    if oid not in baskets:
        baskets[oid] = []
    baskets[oid].append(pname)

col_orders = db["orders"]
col_orders.drop()

batch = []
for oid, items in baskets.items():
    if len(items) >= 2:
        meta = orders_meta.get(oid, {})
        batch.append({
            "_id": oid,
            "order_id": oid,
            "user_id": meta.get("user_id"),
            "order_dow": meta.get("order_dow"),
            "order_hour_of_day": meta.get("order_hour_of_day"),
            "items": items
        })
        if len(batch) >= 10000:
            col_orders.insert_many(batch)
            batch = []

if batch:
    col_orders.insert_many(batch)

col_orders.create_index("order_id")
print(f"[+] Đã nạp thành công {col_orders.count_documents({}):,} giỏ hàng vào MongoDB!")

### Bước 4: Khởi tạo Apache Spark & Huấn luyện mô hình FP-Growth

In [ ]:
import json
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth

# 1. Khởi tạo Spark Session tối ưu cho máy ảo Colab (8GB Memory)
spark = SparkSession.builder \
    .appName("Instacart_FPGrowth_Colab") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Apache Spark Session đã sẵn sàng! Version:", spark.version)

# 2. Kéo dữ liệu từ MongoDB sang định dạng nạp nhanh cho Spark
cursor = col_orders.find({}, {"_id": 0, "order_id": 1, "items": 1})
with open("baskets.jsonl", "w", encoding="utf-8") as f:
    for doc in cursor:
        f.write(json.dumps(doc) + "\n")

df_spark = spark.read.json("baskets.jsonl")
print(f"Số lượng giỏ hàng nạp vào Spark DataFrame: {df_spark.count():,}")
df_spark.show(5, truncate=80)

# 3. Huấn luyện FP-Growth với minSupport = 0.003 (0.3%) và minConfidence = 0.15 (15%)
t_start = time.time()
fp_growth = FPGrowth(itemsCol="items", minSupport=0.003, minConfidence=0.15)
model = fp_growth.fit(df_spark)
print(f"[+] Huấn luyện FP-Growth hoàn tất trong {time.time() - t_start:.2f} giây!")

### Bước 5: Đánh giá Kết quả & Lưu Luật kết hợp vào MongoDB và Google Drive

In [ ]:
# 1. Hiển thị Top 10 Tập phổ biến (Frequent Itemsets)
print("=== TOP 10 TẬP MẶT HÀNG PHỔ BIẾN NHẤT ===")
model.freqItemsets.sort("freq", ascending=False).show(10, truncate=False)

# 2. Khai phá các luật kết hợp mạnh (Lift > 1.2)
rules = model.associationRules
strong_rules = rules.filter(rules["lift"] > 1.2).sort("lift", ascending=False)
print(f"\n=== TỔNG SỐ LUẬT MUA KÈM MẠNH (LIFT > 1.2): {strong_rules.count():,} luật ===")
strong_rules.show(15, truncate=False)

# 3. Lưu vào collection 'recommendation_rules' trên MongoDB
col_rules = db["recommendation_rules"]
col_rules.drop()

rules_data = []
for r in strong_rules.collect():
    rules_data.append({
        "antecedent": list(r["antecedent"]),
        "consequent": list(r["consequent"]),
        "confidence": float(r["confidence"]),
        "lift": float(r["lift"]),
        "support": float(r["support"])
    })

col_rules.insert_many(rules_data)
col_rules.create_index("antecedent")
print(f"[+] Đã lưu {len(rules_data):,} luật vào MongoDB collection 'recommendation_rules'!")

# 4. Xuất file JSON lưu thẳng vào Google Drive (để lưu trữ vĩnh viễn và đồng bộ với Web App local)
output_json_path = os.path.join(DATA_DIR, "recommendation_rules.json")
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(rules_data, f, ensure_ascii=False, indent=2, default=str)
print(f"[+] Đã xuất và lưu file kết quả vào Google Drive: {output_json_path}")

### Bước 6: Trải nghiệm thử nghiệm hàm Gợi ý sản phẩm mua kèm (Live Test)

In [ ]:
def recommend_products(cart_items, top_k=5):
    """
    Hàm gợi ý sản phẩm mua kèm dựa trên giỏ hàng
    Điều kiện: antecedent ⊆ cart_items, loại trừ sản phẩm đã có trong giỏ
    """
    cart_set = set(cart_items)
    cursor = col_rules.find({"antecedent": {"$in": cart_items}}).sort("lift", -1)
    
    recommendations = {}
    for rule in cursor:
        ant_set = set(rule["antecedent"])
        if ant_set.issubset(cart_set):
            for item in rule["consequent"]:
                if item not in cart_set:
                    if item not in recommendations or rule["lift"] > recommendations[item]["lift"]:
                        recommendations[item] = rule
                        
    sorted_recs = sorted(recommendations.values(), key=lambda x: x["lift"], reverse=True)
    return sorted_recs[:top_k]

# Giả lập kịch bản người dùng chọn đồ vào giỏ hàng:
test_cart = ["Yellow Bell Pepper", "Banana"]
results = recommend_products(test_cart)

print(f"Khách hàng chọn vào giỏ: {test_cart}")
print("-" * 60)
print("HỆ THỐNG GỢI Ý CÁC SẢN PHẨM MUA KÈM TIỀM NĂNG NHẤT:")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['consequent'][0]:<28} | Lift: {r['lift']:.2f}x | Confidence: {r['confidence']*100:.1f}%")